# Этап 1: вычитание фона

Нажмите **Run All**, затем подставьте свой файл и запустите ещё раз.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import plotly.graph_objects as go
from ramancore.io import read_spectrum
from ramancore.preprocess import crop, remove_spikes, subtract_baseline

In [ ]:
my_file = "../data/examples/synthetic_si.txt"  # ← подставьте свой файл
x, y, meta = read_spectrum(my_file)
x, y = crop(x, y, 100, 3200)
y = remove_spikes(y)
print("Точек в спектре:", len(x))

## 1. Сравниваем методы

Фон должен идти по «подножию» спектра и не залезать под пики.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, name="спектр"))
for method in ["arpls", "asls", "poly"]:
    corrected, baseline = subtract_baseline(x, y, method=method)
    fig.add_trace(go.Scatter(x=x, y=baseline, name=f"фон {method}", line=dict(dash="dash")))
fig.update_layout(xaxis_title="Рамановский сдвиг, см⁻¹", yaxis_title="Интенсивность")
fig.show()

## 2. Подбираем lam для arpls

Маленькое lam — фон гибкий и может «въехать» в широкие пики; большое — фон жёсткий и не успевает за изгибами.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, name="спектр"))
for lam in [1e3, 1e5, 1e7]:   # можно поменять
    corrected, baseline = subtract_baseline(x, y, method="arpls", lam=lam)
    fig.add_trace(go.Scatter(x=x, y=baseline, name=f"lam = {lam:.0e}", line=dict(dash="dash")))
fig.update_layout(xaxis_title="Рамановский сдвиг, см⁻¹", yaxis_title="Интенсивность")
fig.show()

## 3. Результат после вычитания

In [ ]:
method, lam = "arpls", 1e5   # ← выбранные вами значения
corrected, baseline = subtract_baseline(x, y, method=method, lam=lam)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=corrected, name="спектр без фона"))
fig.add_hline(y=0, line_dash="dot")   # после вычитания подножие должно лежать около нуля
fig.update_layout(xaxis_title="Рамановский сдвиг, см⁻¹", yaxis_title="Интенсивность")
fig.show()